## Phase 1: Check and Install Dependencies

In [1]:
# First, check current Python version
import sys
print(f"Python version: {sys.version}")
print(f"Executable: {sys.executable}")

# Check if we have the right packages
required_packages = {
    'torch': 'torch',
    'transformers': 'transformers',
    'seqeval': 'seqeval',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn'
}

missing = []
for pkg_name, display_name in required_packages.items():
    try:
        __import__(pkg_name)
        print(f"✓ {display_name} installed")
    except ImportError:
        print(f"✗ {display_name} NOT installed")
        missing.append(display_name)

if missing:
    print(f"\nMissing packages: {', '.join(missing)}")
    print("\n💡 Run the cell below to install them!")
else:
    print("\n✓ All dependencies are installed!")

Python version: 3.10.18 (main, Jun  5 2025, 08:13:51) [Clang 14.0.6 ]
Executable: /Users/aida/miniconda3/envs/deep_disfluency/bin/python3.10
✓ torch installed
✓ transformers installed
✓ seqeval installed
✓ numpy installed
✓ pandas installed
✓ scikit-learn installed

✓ All dependencies are installed!


In [2]:
# Install missing dependencies
# Uncomment and run if needed

import subprocess
import sys

print("Installing dependencies...")
packages = [
    "numpy",
    "pandas",
    "pytest",
    "scikit-learn",
]

for pkg in packages:
    print(f"Installing {pkg}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("\nInstalling PyTorch (CPU)...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "torch", "torchvision", "torchaudio",
    "--index-url", "https://download.pytorch.org/whl/cpu"
])

print("\nInstalling transformers and seqeval...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers[torch]", "seqeval"])

print("\n✓ All dependencies installed!")

Installing dependencies...
Installing numpy...
Installing pandas...
Installing pytest...
Installing scikit-learn...

Installing PyTorch (CPU)...

Installing transformers and seqeval...

✓ All dependencies installed!


## Phase 2: Run Integration Tests

In [3]:
# Verify transformer tagger can be imported and instantiated
import sys
sys.path.insert(0, '/Users/aida/Desktop/code/deep_disfluency')

from deep_disfluency.tagger.transformer_tagger import TransformerDisfluencyTagger

print("✓ Successfully imported TransformerDisfluencyTagger")

# Try to create an instance
try:
    tagger = TransformerDisfluencyTagger(
        model_name="distilbert-base-uncased",
        num_labels=9,
        device="cpu",
        context_window=10
    )
    print("✓ Successfully instantiated DistilBERT tagger")
    print(f"  Model: {tagger.model_name}")
    print(f"  Device: {tagger.device}")
    print(f"  Context window: {tagger.context_window}")
except Exception as e:
    print(f"✗ Error instantiating tagger: {e}")
    import traceback
    traceback.print_exc()

✓ Successfully imported TransformerDisfluencyTagger


2025-11-21 08:04:59.236293: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Successfully instantiated DistilBERT tagger
  Model: distilbert-base-uncased
  Device: cpu
  Context window: 10


In [7]:
# Run the integration test suite
import subprocess
import sys

test_file = "/Users/aida/Desktop/code/deep_disfluency/deep_disfluency/tagger/test_deep_tagger_with_transformer.py"

print("Running transformer integration tests...\n")
result = subprocess.run(
    [sys.executable, "-m", "pytest", test_file, "-v"],
    cwd="/Users/aida/Desktop/code/deep_disfluency",
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

if result.returncode == 0:
    print("\n✓ All integration tests passed!")
else:
    print(f"\n✗ Tests failed with exit code {result.returncode}")

Running transformer integration tests...

============================= test session starts ==============================
platform darwin -- Python 3.10.18, pytest-9.0.1, pluggy-1.6.0 -- /Users/aida/miniconda3/envs/deep_disfluency/bin/python3.10
cachedir: .pytest_cache
rootdir: /Users/aida/Desktop/code/deep_disfluency
plugins: anyio-4.11.0
collecting ... collected 0 items / 1 error

==================================== ERRORS ====================================
_ ERROR collecting deep_disfluency/tagger/test_deep_tagger_with_transformer.py _
../../../miniconda3/envs/deep_disfluency/lib/python3.10/site-packages/_pytest/python.py:507: in importtestmodule
    mod = import_path(
../../../miniconda3/envs/deep_disfluency/lib/python3.10/site-packages/_pytest/pathlib.py:587: in import_path
    importlib.import_module(module_name)
../../../miniconda3/envs/deep_disfluency/lib/python3.10/importlib/__init__.py:126: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
<

## Phase 3: Inspect Training Data Format

In [ ]:
import pandas as pd

# Check the training data format
train_file = "/Users/aida/Desktop/code/deep_disfluency/deep_disfluency/data/disfluency_detection/switchboard/swbd_disf_train_1_data_timings.csv"

print(f"Training data file: {train_file}\n")
print("First 20 lines:")
print("="*80)

with open(train_file, 'r') as f:
    for i, line in enumerate(f):
        if i >= 20:
            break
        print(line.rstrip())

print("="*80)

# Count examples
with open(train_file, 'r') as f:
    lines = f.readlines()
    
print(f"\nTotal lines: {len(lines)}")

## Phase 4: Train DistilBERT on Training Data

⏱️ **Estimated time: 30-60 minutes on CPU**

In [ ]:
import subprocess
import sys
import os

train_file = "/Users/aida/Desktop/code/deep_disfluency/deep_disfluency/data/disfluency_detection/switchboard/swbd_disf_train_1_data_timings.csv"
output_dir = "/Users/aida/Desktop/code/deep_disfluency/experiments/transformer_swbd"

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

print(f"Training DistilBERT tagger...")
print(f"Training file: {train_file}")
print(f"Output directory: {output_dir}")
print(f"\nStarting training (this will take 30-60 minutes on CPU)...\n")

result = subprocess.run([
    sys.executable, "-m",
    "deep_disfluency.tagger.train_transformer",
    "--train-file", train_file,
    "--output-dir", output_dir,
    "--epochs", "3",
    "--batch-size", "8",
    "--learning-rate", "2e-5"
], cwd="/Users/aida/Desktop/code/deep_disfluency")

if result.returncode == 0:
    print("\n✓ Training completed successfully!")
    # List output files
    import os
    if os.path.exists(output_dir):
        files = os.listdir(output_dir)
        print(f"\nOutput files in {output_dir}:")
        for f in sorted(files):
            print(f"  - {f}")
else:
    print(f"\n✗ Training failed with exit code {result.returncode}")

## Phase 5: Run Predictions on Heldout Data

In [ ]:
import subprocess
import sys
import os

model_checkpoint = "/Users/aida/Desktop/code/deep_disfluency/experiments/transformer_swbd/epoch_2.pt"
heldout_file = "/Users/aida/Desktop/code/deep_disfluency/deep_disfluency/data/disfluency_detection/switchboard/swbd_disf_heldout_partial_data_timings.csv"
output_dir = "/Users/aida/Desktop/code/deep_disfluency/experiments/transformer_swbd/predictions"

os.makedirs(output_dir, exist_ok=True)

# Check if model checkpoint exists
if not os.path.exists(model_checkpoint):
    print(f"⚠️  Model checkpoint not found: {model_checkpoint}")
    print(f"\nAvailable checkpoints:")
    exp_dir = "/Users/aida/Desktop/code/deep_disfluency/experiments/transformer_swbd"
    if os.path.exists(exp_dir):
        for f in os.listdir(exp_dir):
            print(f"  - {f}")
else:
    print(f"Running predictions on heldout data...")
    print(f"Model: {model_checkpoint}")
    print(f"Input: {heldout_file}")
    print(f"Output directory: {output_dir}\n")
    
    result = subprocess.run([
        sys.executable, "-m",
        "deep_disfluency.tagger.predict_transformer",
        "--model-checkpoint", model_checkpoint,
        "--input", heldout_file,
        "--output", output_dir
    ], cwd="/Users/aida/Desktop/code/deep_disfluency")
    
    if result.returncode == 0:
        print("\n✓ Prediction completed successfully!")
        # List output files
        if os.path.exists(output_dir):
            files = os.listdir(output_dir)
            print(f"\nOutput files in {output_dir}:")
            for f in sorted(files):
                print(f"  - {f}")
    else:
        print(f"\n✗ Prediction failed with exit code {result.returncode}")

## Phase 6: Run Predictions on Test Data

In [ ]:
import subprocess
import sys
import os

model_checkpoint = "/Users/aida/Desktop/code/deep_disfluency/experiments/transformer_swbd/epoch_2.pt"
test_file = "/Users/aida/Desktop/code/deep_disfluency/deep_disfluency/data/disfluency_detection/switchboard/swbd_disf_test_partial_data_timings.csv"
output_dir = "/Users/aida/Desktop/code/deep_disfluency/experiments/transformer_swbd/predictions"

os.makedirs(output_dir, exist_ok=True)

# Check if model checkpoint exists
if not os.path.exists(model_checkpoint):
    print(f"⚠️  Model checkpoint not found: {model_checkpoint}")
else:
    print(f"Running predictions on test data...")
    print(f"Model: {model_checkpoint}")
    print(f"Input: {test_file}")
    print(f"Output directory: {output_dir}\n")
    
    result = subprocess.run([
        sys.executable, "-m",
        "deep_disfluency.tagger.predict_transformer",
        "--model-checkpoint", model_checkpoint,
        "--input", test_file,
        "--output", output_dir
    ], cwd="/Users/aida/Desktop/code/deep_disfluency")
    
    if result.returncode == 0:
        print("\n✓ Test prediction completed successfully!")
    else:
        print(f"\n✗ Test prediction failed with exit code {result.returncode}")

## Phase 7: Next Steps

After training and prediction, you should:

1. **Verify output files**: Check that increco-format output files were created
2. **Run evaluation**: Use the ACL_2026.ipynb notebook to evaluate results against baselines
3. **Integrate into DeepDisfluencyTagger**: Modify `deep_tagger.py` to support the transformer backend
4. **Test in demos**: Run demo notebooks to verify live ASR integration

### Expected Output Files

- `experiments/transformer_swbd/epoch_0.pt` - Checkpoint after epoch 1
- `experiments/transformer_swbd/epoch_1.pt` - Checkpoint after epoch 2
- `experiments/transformer_swbd/epoch_2.pt` - Checkpoint after epoch 3 (final)
- `experiments/transformer_swbd/predictions/swbd_disf_heldout_partial_data_output_increco.text`
- `experiments/transformer_swbd/predictions/swbd_disf_test_partial_data_output_increco.text`
- `experiments/transformer_swbd/predictions/swbd_disf_heldout_partial_data_output_final.text`
- `experiments/transformer_swbd/predictions/swbd_disf_test_partial_data_output_final.text`

In [ ]:
# Verify all expected outputs exist
import os

expected_files = [
    "/Users/aida/Desktop/code/deep_disfluency/experiments/transformer_swbd/epoch_2.pt",
    "/Users/aida/Desktop/code/deep_disfluency/experiments/transformer_swbd/predictions/swbd_disf_heldout_partial_data_output_increco.text",
    "/Users/aida/Desktop/code/deep_disfluency/experiments/transformer_swbd/predictions/swbd_disf_test_partial_data_output_increco.text",
]

print("Checking for expected output files:\n")
all_exist = True
for f in expected_files:
    exists = os.path.exists(f)
    status = "✓" if exists else "✗"
    print(f"{status} {f}")
    if not exists:
        all_exist = False

if all_exist:
    print("\n✓ All expected files created!")
else:
    print("\n⚠️  Some files are still missing. Make sure to run training and predictions above.")